In [ ]:
%pip install paho-mqtt

In [ ]:
import carla
import math
import random
import time
import numpy as np
import os
import json
import threading
import queue as image_queue_module
import paho.mqtt.client as mqtt

In [ ]:
client = carla.Client("localhost", 2000)
client.set_timeout(10.0)
world = client.get_world()

traffic_manager = client.get_trafficmanager(8000)
traffic_manager.set_synchronous_mode(True)

original_settings = world.get_settings()
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05
settings.substepping = True
settings.max_substep_delta_time = 0.01
settings.max_substeps = 10
settings.no_rendering_mode = False
world.apply_settings(settings)

for _ in range(2):
    actors_to_destroy = []
    actors_to_destroy.extend(world.get_actors().filter('sensor.*'))   
    actors_to_destroy.extend(world.get_actors().filter('vehicle.*'))
    actors_to_destroy.extend(world.get_actors().filter('walker.*'))
    
    for actor in actors_to_destroy:
        if actor is not None and actor.is_alive:
            try:
                actor.destroy()
            except RuntimeError:
                pass
            
    world.tick()
    world.tick()

time.sleep(0.5)

carla_map = world.get_map()
spectator = world.get_spectator()

print("Map:", carla_map.name)
print("Synchronous mode:", world.get_settings().synchronous_mode)
print("Fixed delta seconds:", world.get_settings().fixed_delta_seconds)

In [ ]:
v2x_event     = threading.Event()
v2x_sent_flag = threading.Event() 
v2x_time_sent = 0.0
NETWORK_DELAY = 1.5  #seconds

def on_mqtt_message(client, userdata, msg, properties = None):
    if msg.topic == "carla/svs/8/v2x/warning":
        if  msg.payload.decode("utf-8") == "PEDESTRIAN_DETECTED":
            v2x_event.set()

mqtt_client = mqtt.Client(
    callback_api_version=mqtt.CallbackAPIVersion.VERSION2, 
    client_id="Tesla_Ego"
)
mqtt_client.on_message = on_mqtt_message

try:
    mqtt_client.connect("test.mosquitto.org", 1883, 60)
    mqtt_client.subscribe("carla/svs/8/v2x/warning")
    mqtt_client.loop_start()
    time.sleep(0.5)
    if not mqtt_client.is_connected():
        print("[WARN] MQTT non connesso — V2X simulato localmente")
except Exception as e:
    print(f"Error MQTT: {e}.")

In [ ]:
def safe_destroy(actors):
    for actor in actors:
        if actor is None:
            continue
        try:
            actor.destroy()
        except RuntimeError:
            pass

def angle_diff_deg(a, b):
    return (a - b + 180.0) % 360.0 - 180.0


def speed_kmh(vehicle):
    v = vehicle.get_velocity()
    return 3.6 * math.sqrt(v.x * v.x + v.y * v.y + v.z * v.z)


def move_spectator_to(transform, spectator, distance=14.0, z=4.5, pitch=-16.0):
    yaw = math.radians(transform.rotation.yaw)
    back = carla.Location(x=-distance * math.cos(yaw), y=-distance * math.sin(yaw), z=z)
    spectator.set_transform(
        carla.Transform(
            transform.location + back,
            carla.Rotation(pitch=pitch, yaw=transform.rotation.yaw),
        )
    )


def pick_vehicle_bp(world, role_name, preferred_ids):
    lib = world.get_blueprint_library()
    bp = None

    for bp_id in preferred_ids:
        try:
            bp = lib.find(bp_id)
            break
        except RuntimeError:
            continue

    if bp is None:
        fallback = []
        for cand in lib.filter("vehicle.*"):
            if cand.has_attribute("number_of_wheels") and cand.get_attribute("number_of_wheels").as_int() != 4:
                continue
            fallback.append(cand)
        if not fallback:
            raise RuntimeError("No suitable vehicle blueprint found")
        bp = random.choice(fallback)

    if bp.has_attribute("color"):
        colors = list(bp.get_attribute("color").recommended_values)
        if colors:
            bp.set_attribute("color", random.choice(colors))

    if bp.has_attribute("role_name"):
        bp.set_attribute("role_name", role_name)

    return bp


def find_straight_start_waypoints(world, straight_len=95.0, step=2.0, max_yaw_step=6.0, max_candidates=120):
    m = world.get_map()
    spawn_points = m.get_spawn_points()
    random.shuffle(spawn_points)

    candidates = []
    for sp in spawn_points:
        wp0 = m.get_waypoint(sp.location, project_to_road=True, lane_type=carla.LaneType.Driving)
        if wp0 is None:
            continue

        prev = wp0
        traveled = 0.0
        valid = True

        while traveled < straight_len:
            nxt_list = prev.next(step)
            if not nxt_list:
                valid = False
                break

            nxt = min(
                nxt_list,
                key=lambda w: abs(angle_diff_deg(w.transform.rotation.yaw, prev.transform.rotation.yaw)),
            )

            yaw_jump = abs(angle_diff_deg(nxt.transform.rotation.yaw, prev.transform.rotation.yaw))
            if yaw_jump > max_yaw_step:
                valid = False
                break

            if nxt.road_id != wp0.road_id or nxt.lane_id != wp0.lane_id:
                valid = False
                break

            prev = nxt
            traveled += step

        if valid:
            candidates.append(wp0)
        if len(candidates) >= max_candidates:
            break

    return candidates


def spawn_vehicle_pair_straight(world, gap_candidates=(40.0, 45.0, 50.0, 55.0)):

    ego_bp = pick_vehicle_bp(world, "lab5_ego", preferred_ids=("vehicle.tesla.model3", "vehicle.audi.tt"))
    target_bp = pick_vehicle_bp(
        world,
        "lab5_target",
        preferred_ids=("vehicle.lincoln.mkz_2020", "vehicle.mercedes.coupe", "vehicle.tesla.model3"),
    )

    starts = find_straight_start_waypoints(world)
    if not starts:
        raise RuntimeError("No straight lane candidates found on this map")

    for start_wp in starts:
        ego_tf = carla.Transform(start_wp.transform.location + carla.Location(z=0.5), start_wp.transform.rotation)
        ego = world.try_spawn_actor(ego_bp, ego_tf)
        if ego is None:
            continue

        ego.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0))

        target_actor = None
        target_wp = None

        for gap in gap_candidates:
            options = start_wp.next(gap)
            options = [
                w
                for w in options
                if w.road_id == start_wp.road_id
                and w.lane_id == start_wp.lane_id
                and abs(angle_diff_deg(w.transform.rotation.yaw, start_wp.transform.rotation.yaw)) < 8.0
            ]
            options.sort(key=lambda w: abs(angle_diff_deg(w.transform.rotation.yaw, start_wp.transform.rotation.yaw)))

            for cand in options:
                tf = carla.Transform(cand.transform.location + carla.Location(z=0.3), cand.transform.rotation)
                actor = world.try_spawn_actor(target_bp, tf)
                if actor is not None:
                    target_actor = actor
                    target_wp = cand
                    break

            if target_actor is not None:
                break

        if target_actor is not None:
            target_actor.set_autopilot(False)
            target_actor.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0, hand_brake=True))
            return ego, target_actor, start_wp, target_wp

        ego.destroy()

    raise RuntimeError("Could not spawn two vehicles aligned on the same straight lane")

def spawn_camera(world, attach_to, transform):
    bp = world.get_blueprint_library().find('sensor.camera.rgb')
    bp.set_attribute('image_size_x', '800')
    bp.set_attribute('image_size_y', '600')
    
    bp.set_attribute('sensor_tick', '0.05') 
    
    if bp.has_attribute("role_name"):
        bp.set_attribute("role_name", "forensic_dashcam")
        
    return world.spawn_actor(bp, transform, attach_to=attach_to)

def spawn_radar(
    world,
    attach_to,
    transform,
    horizontal_fov=80.0,
    vertical_fov=5.0,
    points_per_second=10000,
    range_m=100.0,
    tick=0.05,
):
    bp = world.get_blueprint_library().find("sensor.other.radar")
    bp.set_attribute("horizontal_fov", str(horizontal_fov))
    bp.set_attribute("vertical_fov", str(vertical_fov))
    bp.set_attribute("points_per_second", str(points_per_second))
    bp.set_attribute("range", str(range_m))
    bp.set_attribute("sensor_tick", str(tick))
    if bp.has_attribute("role_name"):
        bp.set_attribute("role_name", "forensic_mrr_radar")
    return world.spawn_actor(bp, transform, attach_to=attach_to)


def draw_radar_points(world, radar_data, sample_size=200): 
    detections = list(radar_data)
    if not detections:
        return

    step = max(1, len(detections) // sample_size)
    for det in detections[::step]:
        fw = carla.Vector3D(x=det.depth)
        rot = carla.Rotation(yaw=math.degrees(det.azimuth), pitch=math.degrees(det.altitude))
        carla.Transform(carla.Location(), rot).transform(fw)
        point = radar_data.transform.location + fw
        
        color = carla.Color(255, 180, 0) if abs(det.velocity) > 1.0 else carla.Color(130, 210, 255)
        world.debug.draw_point(point, size=0.055, color=color, life_time=0.06)

In [ ]:
class FrontRadarTracker:
    def __init__(self, azimuth_limit_deg=40.0, altitude_limit_deg=4.0, min_depth_m=0.2):
        self.min_depth_m = min_depth_m
        self.distance_m = None
        self.closing_speed_mps = None
        self.ttc_s = None
        self.front_count = 0

    def update(self, filtered_radar_points):
        self.front_count = len(filtered_radar_points)
        
        if not filtered_radar_points:
            self.distance_m = None
            self.closing_speed_mps = None
            self.ttc_s = None
            return

        depths = np.array([p["x"] for p in filtered_radar_points if p["x"] >= self.min_depth_m], dtype=np.float32)
        velocities = np.array([abs(p["rel_velocity"]) for p in filtered_radar_points if p["x"] >= self.min_depth_m], dtype=np.float32)

        if len(depths) == 0:
            self.distance_m = None
            self.closing_speed_mps = None
            self.ttc_s = None
            return

        d_raw = float(np.percentile(depths, 15))
        v_raw = float(np.median(velocities))

        if self.distance_m is None:
            self.distance_m = d_raw
        else:
            self.distance_m = 0.78 * self.distance_m + 0.22 * d_raw

        if self.closing_speed_mps is None:
            self.closing_speed_mps = v_raw
        else:
            self.closing_speed_mps = 0.70 * self.closing_speed_mps + 0.30 * v_raw

        if self.closing_speed_mps > 0.25:
            self.ttc_s = self.distance_m / self.closing_speed_mps
        else:
            self.ttc_s = float("inf")

In [ ]:
def filter_detections_in_lane(radar_data, half_lane_width=1.75, sensor_height_m=1.2):
    
    filtered_points = []
    
    for det in radar_data:
        x_front = det.depth * math.cos(det.azimuth) * math.cos(det.altitude)
        y_lateral = det.depth * math.sin(det.azimuth) * math.cos(det.altitude)
        z_height = det.depth * math.sin(det.altitude)
        
        if z_height < -(sensor_height_m - 0.2) or z_height > 1.0:
            continue
            
        if abs(y_lateral) <= half_lane_width:
            filtered_points.append({
                "x": x_front,
                "y": y_lateral,
                "z": z_height,
                "rel_velocity": det.velocity
            })
            
    return filtered_points

In [ ]:
actors = []
ego = None
target = None
radar = None
camera = None

RADAR_PARAMS = {
    "horizontal_fov": 80.0,      
    "vertical_fov": 5.0,
    "points_per_second": 10000,  
    "range_m": 100.0,            
    "tick": 0.05,              
}

DURATION_SECONDS = 30.0
DT = world.get_settings().fixed_delta_seconds

CRUISE_SPEED_KMH = 30.0
CRUISE_THROTTLE = 1.0

SOFT_DIST_M = 4.0
HARD_DIST_M = 2.8
SOFT_TTC_S = 2.0
HARD_TTC_S = 1.0

tracker = FrontRadarTracker(azimuth_limit_deg=40.0, altitude_limit_deg=4.0, min_depth_m=0.2)

v2x_event.clear()
v2x_sent_flag.clear()
v2x_time_sent = 0.0

throttle            = 0.0
brake               = 0.0
hold_brake_mode     = False
hold_counter        = 0
dist_ped            = float('inf')
ped_triggered       = False
ped_fallen          = False
ped_fallen_logged   = False
hard_condition      = False
soft_condition      = False
save_queue          = None
saver_thread        = None
forensic_logs       = []
output_folder       = "dashcam_records"

os.makedirs(output_folder, exist_ok=True)
LOG_INTERVAL = 4 
V2X_ACTIVE_DURATION = 5.0

try:
    starts = find_straight_start_waypoints(world)
    if not starts: raise RuntimeError("No straight road found")
    spawn_tf = carla.Transform(starts[0].transform.location + carla.Location(z=0.3), starts[0].transform.rotation)

    ego_bp = world.get_blueprint_library().find('vehicle.tesla.model3')
    ego = world.spawn_actor(ego_bp, spawn_tf)
    actors.append(ego)

    fwd = spawn_tf.get_forward_vector()
    side = spawn_tf.get_right_vector()

    van_loc = spawn_tf.location + fwd * 30.0 + side * 2.8 
    van_bp = world.get_blueprint_library().find('vehicle.volkswagen.t2')
    target = world.spawn_actor(van_bp, carla.Transform(van_loc + carla.Location(z=0.5), spawn_tf.rotation))
    actors.append(target)

    ped_spawn_loc = van_loc + fwd * 2.8 + side * 1.5
    ped_bp = world.get_blueprint_library().find('walker.pedestrian.0001')
    ped = world.spawn_actor(ped_bp, carla.Transform(ped_spawn_loc + carla.Location(z=1.0), spawn_tf.rotation))
    actors.append(ped)

    radar_tf = carla.Transform(carla.Location(x=2.7, z=1.2))
    radar = spawn_radar(world, ego, radar_tf, **RADAR_PARAMS)
    actors.append(radar)

    radar_state = {"raw": 0, "filtered": 0}
    def on_radar(measurement):
        radar_state["raw"] = len(measurement)
        lane_points = filter_detections_in_lane(measurement)
        radar_state["filtered"] = len(lane_points)
        tracker.update(lane_points)
        draw_radar_points(world, measurement, sample_size=200)

    radar.listen(on_radar)

    cam_tf = carla.Transform(carla.Location(x=1.5, z=2.4), carla.Rotation(pitch=-5.0))
    camera = spawn_camera(world, ego, cam_tf)
    actors.append(camera)

    save_queue = image_queue_module.Queue(maxsize=100)

    def image_saver_worker():
        while True:
            item = save_queue.get()
            if item is None:
                save_queue.task_done()
                break
            image, path = item
            try:
                image.save_to_disk(path)
            except Exception as e:
                print(f"[ImageSaver] Errore salvataggio {path}: {e}")
            finally:
                save_queue.task_done()

    saver_thread = threading.Thread(target=image_saver_worker, daemon=False)
    saver_thread.start()

    def on_camera(image):
        try:
            save_queue.put_nowait(
                (image, f'{output_folder}/frame_{image.frame:06d}.jpg')
            )
        except Exception:
            pass

    camera.listen(on_camera)

    start_gap = ego.get_location().distance(target.get_location())
    print(f"Spawn OK | dist={start_gap:.1f}m | ego speed={speed_kmh(ego):.1f} km/h")

    total_steps = int(DURATION_SECONDS / DT)
    print_step = max(1, int(1.0 / DT))

    traffic_manager.ignore_vehicles_percentage(ego, 100.0)
    traffic_manager.ignore_walkers_percentage(ego, 100.0)
    traffic_manager.vehicle_percentage_speed_difference(ego, -100.0)

    ego.set_autopilot(True)
    autopilot_active    = True
    last_known_steer    = 0.0
    target_tf           = target.get_transform()
    target_loc          = target.get_location()

    for _ in range(20):
        world.tick()

    for step in range(total_steps):

        time_sim_s = step * DT
        d = tracker.distance_m
        ttc = tracker.ttc_s
        v_kmh = speed_kmh(ego)

        ego_loc = ego.get_location()
        ped_current_loc = ped.get_location()

        dist_ped = ego_loc.distance(ped_current_loc)
        dist_target = ego_loc.distance(target_loc)

        if dist_ped < 15.0 and dist_ped > 1.0 and not ped_triggered:
            cross_dir = side * -1.0
            cross_dir.z = 0.0
            control = carla.WalkerControl(direction=cross_dir, speed=3.5)
            ped.apply_control(control)
            ped_triggered = True

            if not v2x_sent_flag.is_set():
                try:
                    mqtt_client.publish("carla/svs/8/v2x/warning", "PEDESTRIAN_DETECTED")
                except Exception:
                    pass
                v2x_sent_flag.set()
                v2x_time_sent = time_sim_s

        if dist_ped <= 4.0 and not ped_fallen and ped_triggered:
            # Pedone investito: set_collisions(False) gestisce la fisica post-caduta
            # Il walker continua a muoversi ma non interagisce più con l'ego
            current_transform = ped.get_transform()
            new_rotation = carla.Rotation(pitch=-90.0, yaw=current_transform.rotation.yaw, roll=0.0)
            new_location = current_transform.location
            new_location.z -= 0.8 
            ped.set_transform(carla.Transform(new_location, new_rotation))
            ped.set_collisions(False)
            ped_fallen = True

        hard_condition = (d is not None and d < HARD_DIST_M) or (ttc is not None and ttc < HARD_TTC_S)
        soft_condition = (d is not None and d < SOFT_DIST_M) or (ttc is not None and ttc < SOFT_TTC_S)

        v2x_condition = (
            v2x_event.is_set() and
            v2x_sent_flag.is_set() and
            v2x_time_sent > 0.0 and
            (time_sim_s - v2x_time_sent) >= NETWORK_DELAY and
            (time_sim_s - v2x_time_sent) <= (NETWORK_DELAY + V2X_ACTIVE_DURATION)
        )

        tgt_throttle = CRUISE_THROTTLE
        tgt_brake    = 0.0
        if hold_brake_mode:
            if d is None or d > 12.0:
                hold_brake_mode = False
                hold_counter = 0
                tgt_throttle = CRUISE_THROTTLE
                tgt_brake = 0.0
            else:
                tgt_throttle = 0.0
                tgt_brake = 1.0
        else:
            if hard_condition:
                tgt_throttle = 0.0
                tgt_brake = 1.0
            elif soft_condition or v2x_condition:  
                tgt_throttle = 0.0
                tgt_brake = 0.40
            else:
                tgt_throttle = CRUISE_THROTTLE if v_kmh < CRUISE_SPEED_KMH else 0.0
                tgt_brake = 0.0

        throttle = 0.20 * throttle + 0.80 * tgt_throttle
        brake = 0.80 * brake + 0.20 * tgt_brake

        if brake > 0.20:
            throttle = 0.0

        if d is not None and d < 10.0 and v_kmh < 0.50:
            hold_counter += 1
        else:
            hold_counter = 0
        if hold_counter > int(2.0 / DT):
            hold_brake_mode = True
            throttle = 0.0
            brake = max(brake, 0.95)

        adas_is_acting = hold_brake_mode or hard_condition or soft_condition or v2x_condition

        last_known_steer = ego.get_control().steer
        if adas_is_acting and autopilot_active:
            ego.set_autopilot(False)
            autopilot_active = False
        elif not adas_is_acting and not autopilot_active:
            ego.set_autopilot(True)
            autopilot_active = True
            throttle = 0.0
            brake    = 0.0 

        if not autopilot_active:
            ego.apply_control(carla.VehicleControl(
                throttle=float(np.clip(throttle, 0.0, 1.0)),
                brake=float(np.clip(brake, 0.0, 1.0)), 
                steer=last_known_steer,
                hand_brake=False
            ))

        ego_tf = ego.get_transform()

        if step % LOG_INTERVAL == 0:
            log_time = round(step * DT, 2)
            system_state = "Braking" if hold_brake_mode else (
                "Critical" if hard_condition else (
                    "Warning" if soft_condition or v2x_condition else "Normal"
                ))

            explanations = []

            time_since_sent = log_time - v2x_time_sent if v2x_time_sent > 0.0 else float('inf')

            # V2X in transito
            if v2x_sent_flag.is_set() and not v2x_event.is_set() and time_since_sent < NETWORK_DELAY * 3:
                explanations.append({
                    "timestamp": log_time,
                    "event": f"V2X message sent by van, awaiting network delivery ({NETWORK_DELAY}s delay)",
                    "type": "communication_delay"
                })

            # V2X ricevuto ma non ancora attivo o scaduto
            if v2x_event.is_set() and not v2x_condition:
                elapsed = log_time - v2x_time_sent
                already_braking = soft_condition or hard_condition
                if elapsed < NETWORK_DELAY:
                    event_text = (
                        f"V2X received but delay ({NETWORK_DELAY}s) not elapsed — "
                        f"{'braking already active via radar' if already_braking else 'braking not yet triggered'}"
                    )
                else:
                    event_text = f"V2X warning expired after {V2X_ACTIVE_DURATION}s — system returned to normal"
                explanations.append({
                    "timestamp": log_time,
                    "event": event_text,
                    "type": "communication_failure"
                })

            # Collisione — una sola volta
            if ped_fallen and not ped_fallen_logged:
                explanations.append({
                    "timestamp": log_time,
                    "event": f"Pedestrian down at dist={dist_ped:.1f}m from ego — collision or near miss",
                    "type": "collision_event"
                })
                ped_fallen_logged = True

            # Occlusione radar
            if radar_state["raw"] > 0 and radar_state["filtered"] == 0 and dist_ped < 20.0:
                explanations.append({
                    "timestamp": log_time,
                    "event": f"Pedestrian occluded by van: {radar_state['raw']} raw radar points, 0 in-lane filtered — hidden hazard undetected",
                    "type": "environment_hazard"
                })

            # TTC critico
            if hard_condition:
                explanations.append({
                    "timestamp": log_time,
                    "event": "Critical Time To Collision threshold breached",
                    "type": "system_activation"
                })

            forensic_logs.append({
                "frame": step,
                "time_sim_s": log_time,
                "system_state": system_state,
                "ego_vehicle": {
                    "position": {"x": round(ego_tf.location.x, 2), "y": round(ego_tf.location.y, 2), "z": round(ego_tf.location.z, 2)},
                    "rotation": {"pitch": round(ego_tf.rotation.pitch, 2), "yaw": round(ego_tf.rotation.yaw, 2), "roll": round(ego_tf.rotation.roll, 2)},
                    "speed_kmh": round(v_kmh, 2),
                    "controls": {"throttle": round(throttle, 2), "brake": round(brake, 2), "steer": round(last_known_steer, 2)}
                },
                "actors": [
                    {
                        "id": "van_volkswagen",
                        "position": {"x": round(target_tf.location.x, 2), "y": round(target_tf.location.y, 2), "z": round(target_tf.location.z, 2)},
                        "distance_to_ego": round(dist_target, 2)
                    },
                    {
                        "id": "pedestrian_hidden",
                        "position": {"x": round(ped_current_loc.x, 2), "y": round(ped_current_loc.y, 2), "z": round(ped_current_loc.z, 2)},
                        "distance_to_ego": round(dist_ped, 2)
                    }
                ],
                "sensors": {
                    "radar_raw_count": radar_state["raw"],
                    "radar_filtered_count": radar_state["filtered"],
                    "estimated_distance_m": round(d, 2) if d is not None else None,
                    "ttc_s": round(ttc, 2) if ttc and math.isfinite(ttc) else None
                },
                "explanations": explanations
            })

        world.debug.draw_string(
            target_tf.location + carla.Location(z=2.1),
            "TARGET",
            life_time=0.08,
            color=carla.Color(0, 255, 255),
        )

        if d is not None:
            ttc_text_dbg = "inf" if (ttc is None or not math.isfinite(ttc)) else f"{ttc:.2f}s"
            world.debug.draw_string(
                ego_loc + carla.Location(z=2.2),
                f"d={d:.2f}m TTC={ttc_text_dbg}",
                life_time=0.08,
                color=carla.Color(255, 220, 0),
            )

        if step % print_step == 0:
            d_text   = "None" if d   is None else f"{d:.2f}"
            ttc_text = "None" if ttc is None else ("inf" if not math.isfinite(ttc) else f"{ttc:.2f}")
            print(
                f"speed={v_kmh:5.2f} km/h | d={d_text:>6}m | ttc={ttc_text:>5}s | "
                f"thr={throttle:.2f} brk={brake:.2f} | "
                f"v2x_sent={v2x_sent_flag.is_set()} "
                f"v2x_recv={v2x_event.is_set()} "
                f"v2x_cond={v2x_condition} "
                f"hold={hold_brake_mode} "
                f"adas={adas_is_acting}"
            )

        move_spectator_to(ego_tf, spectator, distance=14.0, z=4.5, pitch=-16.0)
        world.tick()

finally:
    print("\[Cleanup] Avvio procedura di pulizia e salvataggio...")

    if forensic_logs:
        try:
            with open('forensic_data.json', 'w') as f:
                json.dump(forensic_logs, f, indent=4)
            print(f"[Cleanup] File 'forensic_data.json' salvato con successo ({len(forensic_logs)} frame).")
        except Exception as e:
            print(f"[Cleanup Error] Impossibile salvare il JSON forense: {e}")

    try:
        mqtt_client.loop_stop()
        mqtt_client.disconnect()
        print("[Cleanup] Thread di background MQTT interrotto e disconnesso.")
    except Exception as e:
        print(f"[Cleanup Error] Glitch durante la chiusura di MQTT: {e}")

    if radar is not None:
        try: radar.stop()
        except Exception: pass

    if camera is not None:
        try:
            if save_queue is not None:
                save_queue.put(None)
            if saver_thread is not None:
                saver_thread.join(timeout=10.0)
                if saver_thread.is_alive():
                    print("[WARN] Image saver non ha terminato entro 10s — frame potrebbero mancare")
            camera.stop()
        except Exception: pass

    if ego is not None and ego.is_alive:
        try:
            ego.set_autopilot(False)
        except Exception:
            pass

    if actors:
        try:
            safe_destroy(actors)
            print("[Cleanup] Tutti gli attori rimossi dallo scenario.")
        except Exception as e:
            print(f"[Cleanup Error] Errore durante la rimozione degli attori: {e}")

    try:
        world.tick()
        world.tick()
    except Exception:
        pass

    try:
        traffic_manager.set_synchronous_mode(False)
        settings = world.get_settings()
        settings.synchronous_mode = False
        settings.fixed_delta_seconds = None
        world.apply_settings(settings)
        print("[Cleanup] Server CARLA ripristinato correttamente in modalità Asincrona.")
    except Exception as e:
        print(f"[CRITICAL ERROR] Impossibile ripristinare la modalità asincrona. Il server potrebbe bloccarsi: {e}")
